# Session 12: Unit Testing & AI-Generated Test Suites

> Automate edge case coverage and generate comprehensive test suites using AI.

## 1. FIRST Principles + Test Pyramid

Good tests share five properties — the **FIRST** acronym:

- **Fast:** Tests should run in milliseconds. A test suite that takes 10 minutes to run will be skipped. Fast tests require avoiding I/O (network calls, database reads, file writes) inside unit tests — use mocks instead.
- **Independent:** Each test must set up its own state and not rely on the order of execution or the side-effects of another test. Tests that depend on each other are fragile — a single failure cascades into many.
- **Repeatable:** The same test must produce the same result every time, in any environment (local machine, CI server, Docker container). Avoid depending on the current date, random numbers without a seed, or external services.
- **Self-validating:** A test must produce a clear pass or fail verdict — not a log message you have to read and interpret manually. Use `assert` statements or assertion libraries.
- **Timely:** Tests should be written *alongside* (or before) the code, not as an afterthought. Tests written after the fact often end up testing the implementation rather than the behaviour.

---

**The Test Pyramid** is a model for how to allocate testing effort:

```
        /\
       /E2E\        (few — slow, brittle, expensive)
      /------\
     /Integr. \     (some — test service boundaries)
    /----------\
   /  Unit Tests \  (many — fast, cheap, isolated)
  /______________\
```

- **Unit tests (base):** Test a single function or class in isolation. They are fast, deterministic, and cheap to write. You should have hundreds of these.
- **Integration tests (middle):** Test how two or more components work together (e.g., your service + your database). Slower and require more setup, but catch contract mismatches that unit tests miss.
- **End-to-end tests (top):** Simulate a real user interacting with the full system through the UI or API. Very slow and brittle. Use sparingly for the most critical happy paths.

**Common mistake:** Inverting the pyramid — relying heavily on E2E tests because "they test everything." E2E tests are slow, flaky, and hard to debug. A failing E2E test tells you *something* is wrong — a failing unit test tells you *exactly what* is wrong.

**FIRST:** Fast · Independent · Repeatable · Self-validating · Timely

```
        [E2E Tests]        ← Few, slow, expensive
      [Integration Tests]  ← Some
    [Unit Tests]           ← Many, fast, cheap
```
**Rule of thumb:** 70% unit / 20% integration / 10% E2E

## 2. Testing Design Patterns

Writing tests is a skill in itself. These patterns help you write tests that are readable, maintainable, and reliable.

**AAA — Arrange, Act, Assert**
Every test should be structured in three clearly separated phases:
1. **Arrange:** Set up the inputs, dependencies, and mocks needed for the test.
2. **Act:** Call the single function or method under test.
3. **Assert:** Verify the outcome is what you expected.

This pattern makes tests easy to read and makes it obvious what is being tested. If you find yourself writing multiple "Act" calls in one test, that's a sign you should split it into multiple tests.

---

**Mocking and the Test Doubles vocabulary**
A "test double" is any object that stands in for a real dependency:
- **Mock:** Records calls made to it and lets you assert on them. Use when you want to verify that a function was called with specific arguments (e.g., "was the email service called?").
- **Stub:** Returns a pre-programmed value. Use when the dependency's return value affects the code under test (e.g., stub the payment gateway to return "approved").
- **Fake:** A lightweight real implementation (e.g., an in-memory database). Use for integration tests where you want real behaviour but without the infrastructure.

**When to mock:** Mock at the boundary of your system — external services, databases, message queues. Do *not* mock internal classes you own; that makes tests brittle and tightly coupled to implementation details.

---

**Parametrize: one test, many inputs**
When the same logic should work for multiple inputs, use `@pytest.mark.parametrize` instead of copy-pasting the same test with different values. This keeps tests DRY and makes it easy to add new test cases.

In [ ]:
import unittest
from unittest.mock import MagicMock, patch

class PaymentService:
    def __init__(self, gateway): self.gateway = gateway
    def charge(self, amount: float, card_token: str) -> dict:
        if amount <= 0: raise ValueError('Amount must be positive')
        if amount > 50000: raise ValueError('Exceeds single-transaction limit')
        return self.gateway.process({'amount': amount, 'token': card_token})

class TestPaymentService(unittest.TestCase):
    def setUp(self):
        self.mock_gateway = MagicMock()
        self.service = PaymentService(self.mock_gateway)

    def test_charge_calls_gateway_with_correct_params(self):
        self.mock_gateway.process.return_value = {'status': 'success'}
        result = self.service.charge(100.0, 'tok_abc')
        self.mock_gateway.process.assert_called_once_with({'amount': 100.0, 'token': 'tok_abc'})

    def test_raises_on_zero_amount(self):
        with self.assertRaises(ValueError): self.service.charge(0, 'tok_abc')

    def test_raises_on_negative_amount(self):
        with self.assertRaises(ValueError): self.service.charge(-50, 'tok_abc')

    def test_raises_above_transaction_limit(self):
        with self.assertRaises(ValueError): self.service.charge(99999, 'tok_abc')

unittest.main(argv=[''], exit=False, verbosity=2)

## 💡 Additional Examples: Unit Testing

**Example 1 — Testing at the right level**
A common beginner mistake is testing implementation details instead of behaviour. If you write a test that asserts `discount_calculator._internal_rate == 0.15`, your test will break every time you refactor — even if the behaviour (the final discount amount) stays correct. Test what the function *does*, not how it *does it*: assert on the return value and observable side effects only.

**Example 2 — Property-based testing with Hypothesis**
Standard tests check specific examples you thought of. Property-based testing generates *hundreds* of random inputs and verifies that a property holds for all of them. For example: "for any two positive integers, `add(a, b)` should equal `add(b, a)`" (commutativity). This often finds edge cases (empty strings, zero, negative numbers, maximum integers) that you would never have written manually. Use `hypothesis` in Python for this.

**Example 3 — Coverage as a guide, not a goal**
Line coverage tells you which lines of code were executed during tests. 100% coverage does NOT mean your code is bug-free — you can have 100% coverage with tests that make no assertions. Coverage is useful for identifying *untested* code paths, not for measuring test quality. A more useful metric is mutation testing (e.g., with `mutmut`): it introduces small bugs into your code and checks whether your tests catch them. If they don't, your tests are not asserting on the right things.

In [ ]:
# ─── Example 1: Parametrized Testing — Discount Calculator ───────────────────
import unittest

class DiscountCalculator:
    MAX_DISCOUNT_PCT = 50.0

    def calculate(self, price: float, coupon_code: str | None) -> float:
        if price < 0:
            raise ValueError('Price cannot be negative')
        COUPONS = {'SAVE10': 0.10, 'SAVE25': 0.25, 'HALF': 0.50}
        rate = COUPONS.get(coupon_code, 0)
        if rate > self.MAX_DISCOUNT_PCT / 100:
            raise ValueError('Coupon exceeds maximum allowed discount')
        return round(price * (1 - rate), 2)

class TestDiscountCalculator(unittest.TestCase):
    def setUp(self):
        self.calc = DiscountCalculator()

    # ── Happy path: parametrized via subTest ──────────────────────────────
    def test_discount_rates(self):
        cases = [
            ('No coupon',    100.0, None,      100.0),
            ('10% off',      100.0, 'SAVE10',  90.0),
            ('25% off',      200.0, 'SAVE25',  150.0),
            ('50% off',      80.0,  'HALF',    40.0),
            ('Invalid code', 100.0, 'INVALID', 100.0),  # Unknown coupons → no discount
        ]
        for description, price, coupon, expected in cases:
            with self.subTest(description):
                result = self.calc.calculate(price, coupon)
                self.assertEqual(result, expected, f'Failed: {description}')

    # ── Edge cases ─────────────────────────────────────────────────────────
    def test_zero_price(self):
        self.assertEqual(self.calc.calculate(0, 'SAVE10'), 0.0)

    def test_negative_price_raises(self):
        with self.assertRaises(ValueError) as ctx:
            self.calc.calculate(-10, None)
        self.assertIn('negative', str(ctx.exception).lower())

    def test_rounding(self):
        # 33.33 * 0.25 = 8.3325 → should round to 24.99
        result = self.calc.calculate(33.33, 'SAVE25')
        self.assertIsInstance(result, float)
        self.assertEqual(result, round(33.33 * 0.75, 2))

runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestDiscountCalculator)
runner.run(suite)

# ─── Example 2: Testing with Mock — External HTTP Service ────────────────────
import unittest
from unittest.mock import MagicMock, patch, call

class WeatherAPIClient:
    BASE_URL = 'https://api.weather.io/v1'
    def __init__(self, api_key: str, http_client):
        self.api_key = api_key
        self.http = http_client

    def get_forecast(self, city: str) -> dict:
        response = self.http.get(
            f'{self.BASE_URL}/forecast',
            params={'city': city, 'key': self.api_key},
            timeout=5
        )
        if response.status_code != 200:
            raise RuntimeError(f'Weather API error: {response.status_code}')
        return response.json()

class WeatherDashboard:
    def __init__(self, client: WeatherAPIClient):
        self.client = client

    def get_travel_advice(self, city: str) -> str:
        forecast = self.client.get_forecast(city)
        temp = forecast.get('temperature_c', 20)
        rain = forecast.get('rain_probability', 0)
        if rain > 70: return f'{city}: Bring umbrella! 🌧️ (rain: {rain}%)'
        if temp < 10: return f'{city}: Wear warm clothes! 🧥 ({temp}°C)'
        return f'{city}: Great weather! ☀️ ({temp}°C)'

class TestWeatherDashboard(unittest.TestCase):
    def _make_mock_response(self, data: dict, status: int = 200):
        mock = MagicMock()
        mock.status_code = status
        mock.json.return_value = data
        return mock

    def test_rainy_weather_advice(self):
        mock_http = MagicMock()
        mock_http.get.return_value = self._make_mock_response({'temperature_c': 22, 'rain_probability': 85})
        dashboard = WeatherDashboard(WeatherAPIClient('test-key', mock_http))
        advice = dashboard.get_travel_advice('Hanoi')
        self.assertIn('umbrella', advice)
        self.assertIn('Hanoi', advice)

    def test_cold_weather_advice(self):
        mock_http = MagicMock()
        mock_http.get.return_value = self._make_mock_response({'temperature_c': 5, 'rain_probability': 10})
        dashboard = WeatherDashboard(WeatherAPIClient('test-key', mock_http))
        advice = dashboard.get_travel_advice('Hanoi')
        self.assertIn('warm', advice)

    def test_api_error_raises_runtime_error(self):
        mock_http = MagicMock()
        mock_http.get.return_value = self._make_mock_response({}, status=503)
        dashboard = WeatherDashboard(WeatherAPIClient('test-key', mock_http))
        with self.assertRaises(RuntimeError) as ctx:
            dashboard.get_travel_advice('Hanoi')
        self.assertIn('503', str(ctx.exception))

    def test_api_called_with_correct_params(self):
        mock_http = MagicMock()
        mock_http.get.return_value = self._make_mock_response({'temperature_c': 25, 'rain_probability': 5})
        client = WeatherAPIClient('my-key-123', mock_http)
        client.get_forecast('Ho Chi Minh City')
        mock_http.get.assert_called_once_with(
            'https://api.weather.io/v1/forecast',
            params={'city': 'Ho Chi Minh City', 'key': 'my-key-123'},
            timeout=5
        )

runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestWeatherDashboard)
runner.run(suite)

# ─── Example 3: TDD — Test-Driven Development workflow ───────────────────────
# Step 1: Write tests FIRST (implementation does not exist yet — they will fail)
# Step 2: Write the minimal implementation to make them pass
# Step 3: Refactor while keeping tests green

class PasswordValidator:
    """Demonstrates TDD cycle: write test → implement → refactor."""
    MIN_LENGTH = 8
    def validate(self, password: str) -> tuple[bool, list[str]]:
        errors = []
        import re
        if len(password) < self.MIN_LENGTH:
            errors.append(f'Must be at least {self.MIN_LENGTH} characters')
        if not re.search(r'[A-Z]', password):
            errors.append('Must contain at least one uppercase letter')
        if not re.search(r'[0-9]', password):
            errors.append('Must contain at least one digit')
        if not re.search(r'[!@#$%^&*]', password):
            errors.append('Must contain at least one special character (!@#$%^&*)')
        return len(errors) == 0, errors

class TestPasswordValidator(unittest.TestCase):
    def setUp(self): self.validator = PasswordValidator()

    def test_valid_password_passes(self):
        ok, errors = self.validator.validate('Secure@123')
        self.assertTrue(ok)
        self.assertEqual(errors, [])

    def test_too_short_fails(self):
        ok, errors = self.validator.validate('Ab1!')
        self.assertFalse(ok)
        self.assertTrue(any('8 characters' in e for e in errors))

    def test_no_uppercase_fails(self):
        ok, errors = self.validator.validate('secure@123')
        self.assertFalse(ok)
        self.assertTrue(any('uppercase' in e for e in errors))

    def test_no_digit_fails(self):
        ok, errors = self.validator.validate('Secure@abc')
        self.assertFalse(ok)
        self.assertTrue(any('digit' in e for e in errors))

    def test_no_special_char_fails(self):
        ok, errors = self.validator.validate('Secure1234')
        self.assertFalse(ok)
        self.assertTrue(any('special' in e for e in errors))

    def test_multiple_violations_reported(self):
        ok, errors = self.validator.validate('weak')
        self.assertFalse(ok)
        self.assertGreater(len(errors), 1, 'Should report all violations at once')

runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestPasswordValidator)
runner.run(suite)


## 3. AI Lab: Test Suite Generation

### 🧪 AI Lab / Practice

> **TODO:** Select your most complex service class → Prompt Claude: 'Generate a complete unit test suite for this class. Cover: happy path, all error conditions, boundary values, and mock all external dependencies. Target 90% code coverage.' → Run coverage report: `pytest --cov=src --cov-report=html` → What was your coverage before vs after?

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')